# C · Plazo real por segmento y diseño del piloto (U4)

**Proyecto Final Integrador · Machine Learning I · Universidad EAN**
Integrantes: Claudia Naranjo, Yery Pedraza y Nicolás Ruiz.

El componente C es una **política sugerida, no un modelo** (anteproyecto §4, C). La pregunta
original, *¿qué plazo conviene ofrecer a cada segmento?*, no se puede responder comparando clientes
entre sí: el plazo se negoció con cada uno, y los mejores pagadores probablemente recibieron plazos
más largos. Por eso C se replanteó en dos partes:

- **Plazo real por segmento:** cuántos días tarda de verdad cada segmento en pagar y si mover el
  plazo pactado mueve el pago. Para separar el plazo de la calidad del cliente, se compara **a cada
  cliente consigo mismo**: sus facturas con plazo más largo frente a sus facturas con plazo más
  corto.
- **Piloto de gestión** como ruta de validación: lo que los datos no pueden responder (si una
  gestión distinta por segmento acorta el pago) se prueba asignando clientes al azar.

| Sección | Pregunta |
|---|---|
| §1 Reparto de plazos | ¿Qué plazos tiene hoy cada segmento y cuánto varían? |
| §2 Plazo real | ¿Cuántos días tarda cada segmento en pagar desde la emisión? |
| §3 ¿Mover el plazo mueve el pago? | Dentro del mismo cliente, ¿cuánto cambian el día de pago y el atraso por cada día de plazo? |
| §4 Datos para el piloto | Clientes activos, facturas por cliente y cuánto se parecen las facturas de un mismo cliente |
| §5 Tamaño del piloto | ¿Qué diferencia alcanzaría a detectar el piloto con los clientes que hay? |

Los segmentos son los adoptados con condiciones en R17. **Confidencialidad:** solo cifras
agregadas.

## 0. Configuración

In [1]:
import sys
from pathlib import Path

import matplotlib
import numpy as np
import pandas as pd
import scipy
import sklearn
from scipy.stats import norm

RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "datos_crudos").is_dir())
sys.path.insert(0, str(RAIZ / "src"))

from cartera import datos
from cartera import etiqueta as et
from cartera import muestra as mu
from cartera import segmentacion as sg
from cartera.config import SEMILLA
from cartera.diseno import CORTE, DESDE, FECHA_CORTE

np.random.seed(SEMILLA)
print("Semilla", SEMILLA, "| Python", sys.version.split()[0], "| pandas", pd.__version__,
      "| numpy", np.__version__, "| scikit-learn", sklearn.__version__, "| scipy", scipy.__version__,
      "| matplotlib", matplotlib.__version__)

Semilla 2026 | Python 3.14.5 | pandas 3.0.6 | numpy 2.5.3 | scikit-learn 1.9.1 | scipy 1.18.1 | matplotlib 3.11.2


Los segmentos se reconstruyen igual que en R17: preparación y K-means ajustados con los clientes de
entrenamiento, y los de prueba asignados al centro más cercano. Aquí se usan **los 226 clientes**,
porque la política y el piloto se aplicarían a toda la cartera segmentada, no a una muestra de
evaluación.

In [2]:
from sklearn.cluster import KMeans

saldos, movs = datos.leer_todo()
perfil = sg.perfil(saldos, movs)
ent, prueba = sg.particion(perfil, SEMILLA)
prep = sg.preparacion().fit(ent[sg.VARIABLES])
km = KMeans(sg.K, n_init=sg.N_INIT, random_state=SEMILLA).fit(prep.transform(ent[sg.VARIABLES]))
# Numeración por atraso mediano: 0 = paga a tiempo o casi, 1 = paga tarde
orden = np.argsort([np.median(ent.atraso_mediano.to_numpy()[km.labels_ == g]) for g in range(sg.K)])
renum = np.argsort(orden)
NOMBRES = {0: "paga a tiempo o casi", 1: "paga tarde"}
segmento = pd.concat([pd.Series(renum[km.labels_], index=ent.index),
                      pd.Series(renum[km.predict(prep.transform(prueba[sg.VARIABLES]))], index=prueba.index)]
                     ).map(NOMBRES).rename("segmento")
segmento.value_counts().to_frame("clientes")

,clientes
segmento,
paga a tiempo o casi,171
paga tarde,55


## 1. Reparto de plazos por segmento

Facturas emitidas de 2022-01 a 2025-06 con desenlace conocido al 1/07/2025, la misma ventana con la
que se formaron los segmentos. El plazo es `plazo_dias_observado` (vencimiento menos emisión). Para
cada cliente se toma su plazo más frecuente.

In [3]:
TRAMOS = [-1, 14, 15, 30, np.inf]
ETIQ_TRAMOS = ["menos de 15", "15", "16 a 30", "más de 30"]

ventana = mu.desenlace_conocido(saldos, movs, CORTE)
ventana = ventana[(ventana.fecha >= DESDE) & (ventana.fecha < CORTE)].join(segmento, on="nit_cliente", how="inner")
ventana["tramo_plazo"] = pd.cut(ventana.plazo_dias_observado, TRAMOS, labels=ETIQ_TRAMOS)

por_cliente = ventana.groupby("nit_cliente").agg(
    segmento=("segmento", "first"),
    plazo_modal=("plazo_dias_observado", lambda s: s.mode().iloc[0]),
    plazos_distintos=("plazo_dias_observado", "nunique"))
por_cliente["tramo_modal"] = pd.cut(por_cliente.plazo_modal, TRAMOS, labels=ETIQ_TRAMOS)

facturas = pd.crosstab(ventana.segmento, ventana.tramo_plazo, normalize="index").mul(100).round(1)
facturas.insert(0, "facturas", ventana.groupby("segmento").size())
facturas.columns.name = "% de facturas por plazo (días)"
facturas

% de facturas por plazo (días),facturas,menos de 15,15,16 a 30,más de 30
segmento,,,,,
paga a tiempo o casi,7042,2.2,78.8,10.1,9.0
paga tarde,1550,2.9,80.2,10.4,6.5


In [4]:
clientes = pd.crosstab(por_cliente.segmento, por_cliente.tramo_modal)
clientes.columns.name = "clientes por plazo más frecuente (días)"
clientes["con más de un plazo"] = por_cliente.groupby("segmento").plazos_distintos.apply(lambda s: int((s > 1).sum()))
clientes

clientes por plazo más frecuente (días),menos de 15,15,16 a 30,más de 30,con más de un plazo
segmento,,,,,
paga a tiempo o casi,4,140,2,25,152
paga tarde,0,49,1,5,50


## 2. Plazo real: días desde la emisión hasta el pago

Facturas pagadas de la misma ventana: cerradas antes del 1/07/2025 y con etiqueta.
**Días hasta el pago** = plazo + atraso, donde el atraso se mide hasta el último recibo de caja. Es lo que le importa a tesorería: cuándo entra el dinero, sin depender de dónde quedó el
vencimiento.

In [5]:
def pagadas(desde, hasta, cierre_antes_de):
    cerradas = et.facturas_cerradas(saldos)
    cerradas = cerradas[(cerradas.fecha >= desde) & (cerradas.fecha < hasta)
                        & (cerradas.fecha_cancelacion_registrada < cierre_antes_de)
                        & (cerradas.plazo_dias_observado >= 0)]
    p = cerradas.assign(atraso=et.atraso_etiqueta(cerradas, movs)).dropna(subset=["atraso"])
    p["dias_pago"] = p.plazo_dias_observado + p.atraso
    return p.join(segmento, on="nit_cliente", how="inner")

pag = pagadas(DESDE, CORTE, CORTE)
pag.groupby("segmento").agg(
    clientes=("nit_cliente", "nunique"), facturas=("dias_pago", "size"),
    plazo_mediano=("plazo_dias_observado", "median"),
    dias_pago_mediana=("dias_pago", "median"),
    dias_pago_p75=("dias_pago", lambda s: s.quantile(0.75)),
    pct_pagadas_dentro_del_plazo=("atraso", lambda s: (s <= 0).mean() * 100)).round(1)

,clientes,facturas,plazo_mediano,dias_pago_mediana,dias_pago_p75,pct_pagadas_dentro_del_plazo
segmento,,,,,,
paga a tiempo o casi,171,7088,15.0,28.0,43.0,28.0
paga tarde,55,1494,15.0,72.5,115.8,4.5


## 3. ¿Mover el plazo mueve el pago? Cada cliente frente a sí mismo

Solo entran los clientes con más de un plazo en sus facturas pagadas. A cada factura se le resta el
promedio de su cliente, en el plazo y en el desenlace, y la **pendiente** se calcula sobre esas
diferencias: cuántos días cambia el desenlace por cada día de plazo **dentro del mismo cliente**.
Así, lo que es estable en el cliente (su calidad, su tamaño, su poder de negociación) no entra en la
comparación.

- Si los clientes pagaran exactamente al vencimiento, la pendiente de los días hasta el pago sería
  1 y la del atraso, 0.
- Si pagaran a su propio ritmo, sin mirar el plazo, la de los días hasta el pago sería 0 y la del
  atraso, −1.

El intervalo del 95 % sale de 2.000 remuestreos **por cliente** (bootstrap), porque las facturas de
un mismo cliente no son independientes.

In [6]:
def sumas_por_cliente(d, y):
    x = d.plazo_dias_observado - d.groupby("nit_cliente").plazo_dias_observado.transform("mean")
    yy = d[y] - d.groupby("nit_cliente")[y].transform("mean")
    return pd.DataFrame({"sxy": x * yy, "sxx": x * x, "c": d.nit_cliente}).groupby("c")[["sxy", "sxx"]].sum()

def pendiente_con_ic(d, y, veces=2000):
    s = sumas_por_cliente(d, y).to_numpy()
    rng = np.random.default_rng(SEMILLA)
    boot = [(t := s[rng.integers(0, len(s), len(s))])[:, 0].sum() / t[:, 1].sum() for _ in range(veces)]
    return s[:, 0].sum() / s[:, 1].sum(), *np.percentile(boot, [2.5, 97.5])

con_variacion = pag[pag.groupby("nit_cliente").plazo_dias_observado.transform("nunique") > 1]
filas = []
for nombre, d in [*con_variacion.groupby("segmento"), ("todos", con_variacion)]:
    fila = {"segmento": nombre, "clientes": d.nit_cliente.nunique(), "facturas": len(d)}
    for y, etiqueta in [("dias_pago", "días hasta el pago"), ("atraso", "atraso")]:
        b, lo, hi = pendiente_con_ic(d, y)
        fila |= {f"{etiqueta}: pendiente": b, f"{etiqueta}: IC inferior": lo, f"{etiqueta}: IC superior": hi}
    filas.append(fila)
pendientes = pd.DataFrame(filas).set_index("segmento")
pendientes.round(2)

,clientes,facturas,días hasta el pago: pendiente,días hasta el pago: IC inferior,días hasta el pago: IC superior,atraso: pendiente,atraso: IC inferior,atraso: IC superior
segmento,,,,,,,,
paga a tiempo o casi,152,6797,0.27,0.16,0.46,-0.73,-0.84,-0.54
paga tarde,50,1412,0.38,-0.36,2.60,-0.62,-1.36,1.60
todos,202,8209,0.29,0.07,0.61,-0.71,-0.93,-0.39


Para leerlo en días: lo que cambiarían el pago y el atraso si a un cliente se le pasara de 15 a 30
días de plazo, según la pendiente de cada segmento.

In [7]:
pd.DataFrame({"pago, días más tarde": pendientes["días hasta el pago: pendiente"] * 15,
              "atraso, días de diferencia": pendientes["atraso: pendiente"] * 15}).round(1)

,"pago, días más tarde","atraso, días de diferencia"
segmento,,
paga a tiempo o casi,4.1,-10.9
paga tarde,5.7,-9.3
todos,4.4,-10.6


**Límite de esta comparación.** Controla lo estable en el cliente, pero no lo que cambia de una
factura a otra. El más importante: según el diccionario de datos (§6, punto 9), `fecha_vencimiento` **puede
ser la actual y no la original**. Si a una factura que se iba a atrasar se le dio una prórroga, su
plazo aparece más largo y su atraso más corto, lo que empuja justo hacia una pendiente de pago
cercana a 0. Con estos datos no se puede descartar.

## 4. Datos para dimensionar el piloto de gestión

**Diseño del piloto.** Dentro de un segmento, los clientes activos se reparten **al azar** en dos
grupos del mismo tamaño: uno recibe la gestión nueva y el otro sigue con la gestión actual. La
gestión nueva es distinta en cada segmento, porque cada uno paga a un ritmo distinto (§2):

| Segmento | Gestión nueva | Por qué |
|---|---|---|
| paga a tiempo o casi | **Aviso automático antes del vencimiento**: un correo por factura, `[POR COMPLETAR: cuántos días antes, con el área de cartera]` | Paga en una mediana de 28 días; un recordatorio puede bastar para que pague cerca del vencimiento. Al ser automático, no suma carga a las gestoras |
| paga tarde | **Contacto personal apenas vence** cada factura, hecho por las gestoras | Paga en una mediana de 72,5 días; hay semanas en juego, y un contacto temprano tiene más posibilidades de efecto que un aviso |

Es una **hipótesis**: el piloto mide si la gestión de cada segmento acorta el pago, y cada segmento
se analiza por separado. La unidad que se asigna al azar es el **cliente**, porque la gestión se hace
con el cliente, no con la factura.

**Frente a la gestión de hoy** (anteproyecto §1): hay un correo automático semanal a cada cliente con
saldo pendiente, y el contacto personal apunta a la factura que está por cumplir 30 días de mora. El
contacto apenas vence la factura es, por tanto, nuevo. El correo semanal **no incluye facturas que
todavía no vencen** (según una gestora del área), así que en «paga a tiempo o casi» el piloto compara
«aviso» contra «nada antes del vencimiento».

**Desenlace principal: días hasta el pago** (§2), con las tardías como desenlace secundario. Es lo
que se quiere mover, y una medida continua suele dar más potencia que «tardía sí o no».

**Datos de referencia.** Facturas emitidas del 1/07/2025 al 30/06/2026 (cuatro trimestres completos
posteriores a la ventana de los segmentos):

- **clientes activos**: segmentados con al menos una factura emitida en esos cuatro trimestres;
- **facturas por cliente y trimestre**: el promedio entre los clientes activos;
- **días hasta el pago**: media y desviación estándar en las facturas pagadas al 18/10/2026; las que
  siguen abiertas no entran, así que la media queda algo optimista;
- **tasa base de tardías**: sobre las facturas con desenlace conocido al 18/10/2026;
- **ICC** (correlación intraclase): cuánto se parecen las facturas de un mismo cliente. Con ICC
  alto, cada factura adicional de un cliente aporta poca información nueva y el piloto necesita más
  clientes. Se estima con el estimador de ANOVA de un factor.

In [8]:
HASTA_REF = pd.Timestamp("2026-07-01")
TRIMESTRES_REF = 4

emitidas_ref = saldos[saldos.es_deuda_facturada & (saldos.plazo_dias_observado >= 0)
                      & (saldos.fecha >= CORTE) & (saldos.fecha < HASTA_REF)].join(segmento, on="nit_cliente", how="inner")
conocidas_ref = mu.desenlace_conocido(saldos, movs, FECHA_CORTE)
conocidas_ref = conocidas_ref[(conocidas_ref.fecha >= CORTE) & (conocidas_ref.fecha < HASTA_REF)].join(segmento, on="nit_cliente", how="inner")
pag_ref = pagadas(CORTE, HASTA_REF, FECHA_CORTE)

def icc_anova(grupos):
    # Correlación intraclase con el estimador de ANOVA de un factor
    n_i = grupos.size()
    N, k = n_i.sum(), len(n_i)
    media = grupos.mean()
    total = (media * n_i).sum() / N
    msb = (n_i * (media - total) ** 2).sum() / (k - 1)
    msw = (grupos.var(ddof=0) * n_i).sum() / (N - k)
    n0 = (N - (n_i**2).sum() / N) / (k - 1)
    return (msb - msw) / (msb + (n0 - 1) * msw)

base = {}
for s in NOMBRES.values():
    e, c, p = (d[d.segmento == s] for d in (emitidas_ref, conocidas_ref, pag_ref))
    base[s] = {"clientes segmentados": int((segmento == s).sum()),
               "clientes activos": e.nit_cliente.nunique(),
               "facturas emitidas": len(e),
               "facturas por cliente y trimestre": e.groupby("nit_cliente").size().mean() / TRIMESTRES_REF,
               "% pagadas al 18/10/2026": len(p) / len(e) * 100,
               "días hasta el pago: media": p.dias_pago.mean(),
               "días hasta el pago: desviación": p.dias_pago.std(),
               "ICC días hasta el pago": icc_anova(p.groupby("nit_cliente").dias_pago),
               "tasa base de tardías (%)": c.tardia.mean() * 100,
               "ICC tardías": icc_anova(c.groupby("nit_cliente").tardia)}
base = pd.DataFrame(base).T
base.round(2)

,clientes segmentados,clientes activos,facturas emitidas,facturas por cliente y trimestre,% pagadas al 18/10/2026,días hasta el pago: media,días hasta el pago: desviación,ICC días hasta el pago,tasa base de tardías (%),ICC tardías
paga a tiempo o casi,171.0,122.0,2223.0,4.56,94.42,39.55,34.43,0.51,26.32,0.45
paga tarde,55.0,33.0,539.0,4.08,86.09,70.76,37.89,0.37,78.60,0.29


## 5. Tamaño del piloto

Con todos los clientes activos de un segmento repartidos mitad y mitad, se calcula la **diferencia
mínima detectable**: la más pequeña que el piloto encontraría el 80 % de las veces si de verdad
existiera, con una prueba de dos colas al 5 %. Las facturas de un mismo cliente cuentan menos que
facturas independientes: el **efecto de diseño** es 1 + (m − 1) · ICC, donde m es el número de
facturas por cliente durante el piloto.

Se calcula para un piloto de 3 y de 6 meses. Es una aproximación normal con tamaño de cliente
promedio; como los clientes emiten cantidades muy distintas de facturas, el tamaño real necesario es
algo mayor.

In [9]:
ALFA, POTENCIA = 0.05, 0.80
Z = norm.ppf(1 - ALFA / 2) + norm.ppf(POTENCIA)

def diferencia_minima(varianza, clientes_por_grupo, m, icc):
    return Z * np.sqrt(2 * varianza * (1 + (m - 1) * icc) / (clientes_por_grupo * m))

filas = []
for s, b in base.iterrows():
    por_grupo = int(b["clientes activos"] // 2)
    p = b["tasa base de tardías (%)"] / 100
    for meses in (3, 6):
        m = b["facturas por cliente y trimestre"] * meses / 3
        filas.append({"segmento": s, "meses": meses, "clientes por grupo": por_grupo,
                      "facturas por cliente (m)": m,
                      "días hasta el pago: diferencia mínima (días)":
                          diferencia_minima(b["días hasta el pago: desviación"] ** 2, por_grupo, m, b["ICC días hasta el pago"]),
                      "tardías: diferencia mínima (puntos)":
                          diferencia_minima(p * (1 - p), por_grupo, m, b["ICC tardías"]) * 100})
pd.DataFrame(filas).set_index(["segmento", "meses"]).round(1)

clientes por grupo  facturas por cliente (m)  \
segmento             meses                                                 
paga a tiempo o casi 3                      61                       4.6   
                     6                      61                       9.1   
paga tarde           3                      16                       4.1   
                     6                      16                       8.2   

                            días hasta el pago: diferencia mínima (días)  \
segmento             meses                                                 
paga a tiempo o casi 3                                              13.7   
                     6                                              13.1   
paga tarde           3                                              27.2   
                     6                                              25.1   

                            tardías: diferencia mínima (puntos)  
segmento             meses                                       
paga a tiempo o casi 3                                     16.9  
                     6                                     15.9  
paga tarde           3                                     27.7  
                     6                                     25.0

**Carga de la operación.** Cada factura de un cliente del grupo con gestión nueva pide un contacto:
un aviso automático en «paga a tiempo o casi» y un contacto personal de las gestoras en «paga
tarde». Se estima con las facturas por cliente y trimestre del periodo de referencia y los días
hábiles de lunes a viernes de ese periodo, sin restar festivos; es una aproximación. Solo los
contactos personales se suman al trabajo de las gestoras.

In [10]:
dias_habiles_trimestre = np.busday_count(CORTE.date(), HASTA_REF.date()) / TRIMESTRES_REF
carga = pd.DataFrame({
    "tipo de contacto": {"paga a tiempo o casi": "aviso automático", "paga tarde": "contacto personal"},
    "clientes con gestión nueva": base["clientes activos"] // 2,
    "contactos por trimestre": (base["clientes activos"] // 2) * base["facturas por cliente y trimestre"]})
carga["contactos por día hábil"] = carga["contactos por trimestre"] / dias_habiles_trimestre
carga.round(1)

,tipo de contacto,clientes con gestión nueva,contactos por trimestre,contactos por día hábil
paga a tiempo o casi,aviso automático,61.0,277.9,4.3
paga tarde,contacto personal,16.0,65.3,1.0


## 6. Lectura

1. **Casi no hay variación de plazo entre clientes.** Cerca del 80 % de las facturas de ambos
   segmentos tiene plazo de 15 días, y 189 de los 226 clientes tienen 15 días como plazo más
   frecuente. Hoy la empresa no da condiciones distintas a quien paga tarde.
2. **El plazo real es muy distinto del pactado.** Con el mismo plazo mediano de 15 días, «paga a
   tiempo o casi» paga en una mediana de 28 días desde la emisión (solo el 28 % dentro del plazo) y
   «paga tarde», en 72,5 días (4,5 % dentro del plazo).
3. **Dentro del mismo cliente, el plazo pactado mueve poco el pago.** Por cada día de plazo de más,
   el pago llega 0,29 días después (IC 95 %: 0,07 a 0,61) y el atraso medido baja 0,71 días. Pasar
   a un cliente de 15 a 30 días retrasaría el pago unos 4 días y bajaría el atraso medido unos 11:
   **las tardías bajarían en el papel mucho más de lo que cambia la fecha en que entra el dinero.**
   En «paga a tiempo o casi» el resultado es claro (0,27; IC 0,16 a 0,46). En «paga tarde» el
   intervalo es demasiado ancho para concluir (−0,36 a 2,60), con solo 50 clientes.
4. **El piloto de gestión solo detectaría efectos grandes.** Con todos los clientes activos
   repartidos mitad y mitad, la diferencia mínima detectable en días hasta el pago es de unos 13 a
   14 días en «paga a tiempo o casi» (media de 39,6) y de 25 a 27 en «paga tarde» (media de 70,8):
   cerca de un tercio de la media en ambos. En tardías, de 16 a 17 y de 25 a 28 puntos. Medir en
   días no da mucha más potencia que medir tardías, y alargar el piloto de 3 a 6 meses casi no
   ayuda, porque las facturas de un mismo cliente se parecen mucho (ICC entre 0,29 y 0,51).
5. **La gestión por segmento cabe en la operación.** El piloto prueba un aviso automático antes del
   vencimiento en «paga a tiempo o casi» y un contacto personal apenas vence la factura en «paga
   tarde». Con la mitad de los clientes activos en el grupo con gestión nueva, son unos 278 avisos
   automáticos por trimestre, que no ocupan a las gestoras, y unos 65 contactos personales, cerca de
   1 por día hábil frente a los ≈ 20 clientes diarios que hoy gestionan (anteproyecto §1).

**Consecuencia para C.** Con estos datos, **cambiar el plazo pactado no es una palanca para
acelerar el recaudo**, y medir la cartera solo por tardías premiaría alargar plazos sin que el
dinero llegue antes. Por eso se sugiere seguir el plazo real (días hasta el pago) por segmento. Lo
que los datos no responden, si una gestión distinta por segmento acorta el pago, queda para el
piloto, declarando que con la cartera actual solo detectaría un efecto grande. Límite principal: el
vencimiento puede ser el actual y no el original (§3).